# Wildfire AI Intelligence
## Deep Neural Network (MLP) & Generative AI Risk Prediction Notebook

This notebook loads the 32,000+ Indian Forest Wildfire dataset, performs exploratory data analysis, trains a **Deep Multi-Layer Perceptron (MLP) Neural Network** model evaluates performance metrics, saves model weights, and demonstrates grounded GenAI safety advisory synthesis.

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print('Data Science & Deep Learning Libraries Loaded Successfully!')

Data Science & Deep Learning Libraries Loaded Successfully!


### 1. Load Wildfire Dataset (32,002 Records across 25 Forest Reserves)

In [6]:
df = pd.read_csv('wildfire_combined_dataset_30k.csv')

In [7]:
print(f'Dataset Shape: {df.shape}')

Dataset Shape: (32000, 38)


In [9]:
df.head()

,record_id,date,acq_time,season,data_source,state,region,latitude,longitude,elevation_m,...,human_activity_index,state_forest_cover_percent,state_fire_prone_category,state_annual_fire_incidents_approx,firms_satellite_source,firms_brightness_K,firms_confidence,firms_frp_MW,fire_risk_level,fire_occurred
0,WEW0000001,13-08-2021,03:00,Monsoon,Live_API,Chhattisgarh,Indravati Forest,19.5591,81.4158,1199,...,3.98,41.14,High,38106,VIIRS,314.1,60.0,20.4,Extreme,1
1,WEW0000002,19-02-2025,03:00,Winter,Live_API,Karnataka,Bandipur Forest,11.5818,76.9652,738,...,3.25,20.19,Medium,9500,NaN,NaN,NaN,NaN,Medium,0
2,WEW0000003,20-07-2023,00:15,Monsoon,Historical_Kaggle,Jharkhand,Palamu Forest,24.0163,84.1033,240,...,3.01,29.76,High,9800,NaN,NaN,NaN,NaN,Low,0
3,WEW0000004,18-09-2022,16:45,Monsoon,Live_API,Tamil Nadu,Nilgiri Forest,11.2587,76.7848,609,...,8.18,20.27,Medium,3600,NaN,NaN,NaN,NaN,High,0
4,WEW0000005,02-09-2017,18:30,Monsoon,Live_API,Karnataka,BRT Hills Forest,12.6763,77.6122,617,...,2.22,20.19,Medium,9500,NaN,NaN,NaN,NaN,Medium,0


### 2. Feature Engineering & Preprocessing

In [10]:
feature_cols = ['temperature_C', 'humidity_percent', 'wind_speed_kmh', 
                'ndvi_index', 'soil_moisture_percent', 'drought_fire_weather_index', 
                'human_activity_index']

veg_mapping = {'Dry Deciduous': 1.15, 
               'Scrubland': 1.10, 
               'Plantation': 1.05, 
               'Dense Deciduous': 0.95, 
               'Evergreen': 0.85}

In [11]:
df['veg_multiplier'] = df['vegetation_type'].map(lambda x: veg_mapping.get(x, 1.0))

In [12]:
X = df[feature_cols + ['veg_multiplier']]
y = df['fire_risk_level']

In [13]:
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [14]:
print(f'Train shape: {X_train_scaled.shape}, Test shape: {X_test_scaled.shape}')

Train shape: (25600, 8), Test shape: (6400, 8)


### 3. Train Deep Neural Network (MLP Classifier - No RandomForest)

In [15]:
print('Training Deep Multi-Layer Perceptron (MLP) Neural Network...')

mlp_model = MLPClassifier(hidden_layer_sizes=(128, 64, 32),
                          max_iter=300, random_state=42, 
                          early_stopping=True)

mlp_model.fit(X_train_scaled, y_train)

y_pred = mlp_model.predict(X_test_scaled)
acc = accuracy_score(y_test, y_pred)

Training Deep Multi-Layer Perceptron (MLP) Neural Network...


In [17]:
print(f'Model Accuracy: {acc * 100:.2f}%')

Model Accuracy: 75.50%


In [18]:
print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))


Classification Report:
              precision    recall  f1-score   support

     Extreme       0.94      0.89      0.92      2229
        High       0.58      0.49      0.53      1109
         Low       0.88      0.76      0.81      1630
      Medium       0.55      0.75      0.63      1432

    accuracy                           0.76      6400
   macro avg       0.74      0.72      0.72      6400
weighted avg       0.77      0.76      0.76      6400



### 4. Save Model Weights & Demonstrate GenAI Advisory Synthesis

In [19]:
model_data = {
    'model': mlp_model,
    'scaler': scaler,
    'label_encoder': label_encoder,
    'classes': label_encoder.classes_.tolist(),
    'accuracy': float(acc)
}
joblib.dump(model_data, 'wildfire_ai_model.pkl')

print('Saved trained weights to wildfire_ai_model.pkl!')

Saved trained weights to wildfire_ai_model.pkl!
